In [3]:
import findspark
findspark.init()
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("airline_analytics") \
    .master("spark://172.28.1.2:7077") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.executor.cores", "3") \
    .config("spark.executor.instances", "2") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.4") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.sql.files.maxPartitionBytes", "134217728") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.adaptive.skewJoin.enabled", "true") \
    .config("spark.sql.warehouse.dir", "hdfs://master/user/hive/warehouse") \
    .config("hive.metastore.uris", "thrift://master:9083") \
    .enableHiveSupport() \
    .getOrCreate()

# Verify it worked
# spark.conf.get("spark.sql.catalogImplementation")  # should return "hive"

# # List HDFS root
# files = spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem \
#     .get(spark.sparkContext._jsc.hadoopConfiguration()) \
#     .listStatus(spark.sparkContext._jvm.org.apache.hadoop.fs.Path("/"))

# for f in files:
#     print(f.getPath())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/28 17:56:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/02/28 17:56:13 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/02/28 17:56:13 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
26/02/28 17:56:41 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/02/28 17:56:41 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:

In [38]:
spark.sql('show tables').show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|  default|flights_clean|      false|
|  default|          src|      false|
+---------+-------------+-----------+



26/02/28 10:53:58 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/02/28 10:53:58 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [39]:
# Q1 — Which airlines have the highest average arrival delay and how reliable are they?
q1 = spark.sql("""
    SELECT 
        OP_CARRIER,
        ROUND(AVG(ARR_DELAY), 2)                                           AS avg_arr_delay_mins,
        ROUND(AVG(DEP_DELAY), 2)                                           AS avg_dep_delay_mins,
        COUNT(*)                                                            AS total_flights,
        ROUND(SUM(CASE WHEN ARR_DELAY <= 0 THEN 1 ELSE 0 END) * 100.0 
              / COUNT(*), 2)                                                AS on_time_rate_pct
    FROM flights_clean
    WHERE ARR_DELAY IS NOT NULL
    GROUP BY OP_CARRIER
    ORDER BY avg_arr_delay_mins DESC
""")

print("Q1 — Airline Reliability Scorecard")
q1.toPandas()

Q1 — Airline Reliability Scorecard


,OP_CARRIER,avg_arr_delay_mins,avg_dep_delay_mins,total_flights,on_time_rate_pct
0,G4,548.43,12.88,253657,22.87
1,YX,351.22,15.04,748054,27.39
2,NK,323.28,12.77,846924,41.97
3,AS,225.86,4.59,1989037,55.16
4,F9,181.44,12.37,1077340,45.75
5,UA,175.84,12.06,5563225,56.28
6,OH,173.36,14.18,950834,32.82
7,AA,164.42,10.82,7820623,51.77
8,B6,160.35,13.06,2820052,51.71
9,YV,133.19,9.30,1340560,49.26


In [40]:
# Q2 — What are the top 10 most congested routes by average total delay?
q2 = spark.sql("""
    SELECT 
        ORIGIN,
        DEST,
        CONCAT(ORIGIN, ' → ', DEST)                                        AS route,
        COUNT(*)                                                            AS total_flights,
        ROUND(AVG(DEP_DELAY), 2)                                           AS avg_dep_delay_mins,
        ROUND(AVG(ARR_DELAY), 2)                                           AS avg_arr_delay_mins,
        ROUND(AVG(DEP_DELAY + ARR_DELAY), 2)                               AS avg_total_delay_mins
    FROM flights_clean
    GROUP BY ORIGIN, DEST
    HAVING COUNT(*) > 200
    ORDER BY avg_total_delay_mins DESC
    LIMIT 10
""")

print("Q2 — Top 10 Most Delayed Routes")
q2.toPandas()

Q2 — Top 10 Most Delayed Routes


,ORIGIN,DEST,route,total_flights,avg_dep_delay_mins,avg_arr_delay_mins,avg_total_delay_mins
0,BOS,HNL,BOS → HNL,273,18.05,5095.00,5113.05
1,HNL,BOS,HNL → BOS,274,16.98,5095.00,5111.98
2,SFO,CMH,SFO → CMH,221,25.84,2120.00,2145.84
3,CMH,SFO,CMH → SFO,220,13.07,2120.00,2133.07
4,BUR,BOS,BUR → BOS,489,17.30,2037.44,2042.88
5,SEA,CMH,SEA → CMH,409,25.35,2016.00,2041.35
6,BOS,BUR,BOS → BUR,490,17.33,2030.73,2034.97
7,CMH,SEA,CMH → SEA,407,15.69,2016.00,2031.69
8,SJC,BNA,SJC → BNA,315,11.74,1947.00,1958.74
9,ATL,BUR,ATL → BUR,403,16.89,1941.00,1957.89


In [36]:
spark.stop()

26/02/28 10:17:30 WARN NioEventLoop: Selector.select() returned prematurely 512 times in a row; rebuilding Selector io.netty.channel.nio.SelectedSelectionKeySetSelector@51e015f6.


In [43]:
# Q3: Do flights scheduled to depart later in the day experience more arrival delays?
q3 = spark.sql("""
    SELECT
        CASE
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 500  AND 859  THEN '1. Morning   (05:00-08:59)'
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 900  AND 1159 THEN '2. Mid-Day   (09:00-11:59)'
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 1200 AND 1659 THEN '3. Afternoon (12:00-16:59)'
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 1700 AND 2059 THEN '4. Evening   (17:00-20:59)'
            ELSE                                                       '5. Night     (21:00-04:59)'
        END                                                         AS time_of_day,
        COUNT(*)                                                    AS total_flights,
        ROUND(AVG(DEP_DELAY), 2)                                   AS avg_dep_delay_mins,
        ROUND(AVG(ARR_DELAY), 2)                                   AS avg_arr_delay_mins,
        ROUND(SUM(CASE WHEN ARR_DELAY <= 0 THEN 1 ELSE 0 END) 
              * 100.0 / COUNT(*), 2)                               AS on_time_rate_pct
    FROM flights_clean
    WHERE ARR_DELAY IS NOT NULL
      AND DEP_DELAY IS NOT NULL
      AND CRS_DEP_TIME IS NOT NULL
    GROUP BY
        CASE
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 500  AND 859  THEN '1. Morning   (05:00-08:59)'
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 900  AND 1159 THEN '2. Mid-Day   (09:00-11:59)'
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 1200 AND 1659 THEN '3. Afternoon (12:00-16:59)'
            WHEN CAST(CRS_DEP_TIME AS INT) BETWEEN 1700 AND 2059 THEN '4. Evening   (17:00-20:59)'
            ELSE                                                       '5. Night     (21:00-04:59)'
        END
    ORDER BY time_of_day
""")

print("Q3 — Delay Pattern by Time of Day")
q3.toPandas()

Q3 — Delay Pattern by Time of Day


,time_of_day,total_flights,avg_dep_delay_mins,avg_arr_delay_mins,on_time_rate_pct
0,1. Morning (05:00-08:59),15583251,4.59,113.08,62.01
1,2. Mid-Day (09:00-11:59),13168728,7.63,105.65,57.26
2,3. Afternoon (12:00-16:59),21337509,11.41,106.94,51.83
3,4. Evening (17:00-20:59),15668030,15.08,112.98,48.10
4,5. Night (21:00-04:59),4244733,12.73,170.29,48.77


In [44]:
# Q4: Which airline recovers departure delays best
q4 = spark.sql("""
    SELECT
        OP_CARRIER,
        COUNT(*)                                                        AS total_late_departures,
        SUM(CASE WHEN ARR_DELAY <= 0 THEN 1 ELSE 0 END)               AS arrived_on_time,
        ROUND(SUM(CASE WHEN ARR_DELAY <= 0 THEN 1 ELSE 0 END) 
              * 100.0 / COUNT(*), 2)                                   AS recovery_rate_pct,
        ROUND(AVG(DEP_DELAY), 2)                                       AS avg_dep_delay_mins,
        ROUND(AVG(ARR_DELAY), 2)                                       AS avg_arr_delay_mins,
        ROUND(AVG(DEP_DELAY - ARR_DELAY), 2)                          AS avg_mins_recovered
    FROM flights_clean
    WHERE DEP_DELAY > 15
      AND ARR_DELAY IS NOT NULL
      AND DEP_DELAY IS NOT NULL
    GROUP BY OP_CARRIER
    HAVING COUNT(*) > 500
    ORDER BY recovery_rate_pct DESC
""")

print("Q4 — Which Airlines Recover Departure Delays Best?")
q4.toPandas()

Q4 — Which Airlines Recover Departure Delays Best?


,OP_CARRIER,total_late_departures,arrived_on_time,recovery_rate_pct,avg_dep_delay_mins,avg_arr_delay_mins,avg_mins_recovered
0,CO,119771,7751,6.47,56.56,51.69,4.87
1,VX,70936,3245,4.57,65.00,62.19,2.81
2,UA,1330196,55784,4.19,51.87,458.35,-406.48
3,NW,39489,1446,3.66,55.32,53.84,1.48
4,DL,1629267,57458,3.53,49.88,362.71,-312.83
5,B6,706193,20407,2.89,56.36,333.25,-276.89
6,US,310323,8609,2.77,54.96,53.30,1.66
7,WN,2811179,76529,2.72,46.17,119.63,-73.46
8,AA,1779236,41950,2.36,49.37,401.22,-351.85
9,FL,172194,3761,2.18,57.90,56.66,1.24


In [4]:
spark.stop()